# chemagent — Debugging Notebook

End-to-end pipeline using the consolidated `chemagent_mcp.py` server (16 tools).

**Preferred workflow (data stays on disk)**
```
find_datasets → load_dataset → compute_features → split_dataset
  → train_model (non-blocking) → check_training (poll)
  → plot_classification_results
```
**Shortcut (load/featurize/split synchronously, then trains in background)**
```
job = run_pipeline("data/datasets/...", algorithm="RFC", task="classification")
result = check_training(job["job_id"])   # poll every 15-30 s
```


## 1. Environment setup

In [1]:
import sys
from pathlib import Path

# Resolve workspace root (works whether cwd is notebooks/ or the repo root)
_ws_root = Path.cwd()
if not (_ws_root / "src").exists():
    _ws_root = _ws_root.parent

_servers_dir = _ws_root / "src" / "chemagent" / "servers"

for _p in [str(_ws_root), str(_servers_dir)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

_save_dir = _ws_root / "notebooks" / "debug_outputs"
_save_dir.mkdir(exist_ok=True)

print(f"Workspace root : {_ws_root}")
print(f"Servers dir    : {_servers_dir}")
print(f"Save dir       : {_save_dir}")

Workspace root : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability
Servers dir    : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\src\chemagent\servers
Save dir       : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\notebooks\debug_outputs


## 2. Imports

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

from src.chemagent.servers.chemagent_mcp import (
    # ── Dataset tools (MCP) ────────────────────────────────────
    find_datasets,
    list_featurizers,
    load_dataset,
    dataset_status,
    compute_features,
    split_dataset,
    # ── ML tools (MCP) ─────────────────────────────────────────
    get_ml_info,
    train_model,
    check_training,
    export_predictions,
    
)
from src.chemagent.plots import (
    plot_confusion_matrix,
    plot_roc_curve,
    plot_pr_curve,
    plot_metric_bar,
    plot_class_distribution,
    plot_split_statistics,
)
import joblib, numpy as np, time


c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Discover available options

In [3]:
find_datasets()


{'datasets': ['chembl_activity_data_O00329_P42336.csv',
  'chembl_activity_data_O00329_P48736.csv',
  'chembl_activity_data_P42336_P48736.csv'],
 'count': 3,
 'directory': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\datasets'}

In [4]:
load_dataset('/data/datasets/chembl_activity_data_O00329_P48736.csv')

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'n_samples': 1277,
 'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
 'label_col': 'class_label',
 'has_smiles': True,
 'has_precomputed_features': False,
 'loaded': True,
 'next_step': "Call featurize_dataset(dataset_id='chembl_activity_data_O00329_P48736', method='ECFP', radius=2, n_bits=2048) to compute fingerprints server-side, then split_prepared_dataset().",
 'smiles_sample': ['CC(NC(=O)c1c(N)nn2cccnc12)c1cc2cccc(Cl)c2c(=O)n1-c1ccc(O)cc1',
  'Cc1cccc(NS(=O)(=O)c2ccc(C)c(-c3cnc(N)c(-c4cnn(C)c4)c3)c2)n1',
  'Cc1ccc(S(=O)(=O)NCC(C)(C)O)cc1-c1cnc(N)c(-c2ncco2)c1']}

In [5]:
list_featurizers()

{'ECFP': {'parameters': {'n_bits': '2048',
   'radius': '2',
   'sparse': 'False',
   'return_bit_info': 'False'},
  'description': 'Generate ECFP (Morgan) bit-vector fingerprints from SMILES strings.'},
 'MACCS': {'parameters': {},
  'description': 'Generate 166-bit MACCS structural-key fingerprints from SMILES strings.'}}

In [6]:
# Returns algorithms + hyperparameter grids + recommended metrics in one call
get_ml_info()

{'algorithms': {'RFR': {'name': 'Random Forest Regressor',
   'task_type': 'regression',
   'hyperparameters': {'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]},
   'supports_multiclass': False,
   'description': 'Ensemble of decision trees for regression tasks'},
  'RFC': {'name': 'Random Forest Classifier',
   'task_type': 'classification',
   'hyperparameters': {'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]},
   'supports_multiclass': True,
   'description': 'Ensemble of decision trees for classification, handles multi-class'},
  'SVC': {'name': 'Support Vector Classifier',
   'task_type': 'classification',
   'hyperparameters': {'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']},
   'supports_multiclass': True,
   'description': 'SVM classifier with RBF/linear kern

## 4. Load dataset

In [7]:
dataset_info = load_dataset("data/datasets/chembl_activity_data_O00329_P48736.csv")
dataset_info

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'n_samples': 1277,
 'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
 'label_col': 'class_label',
 'has_smiles': True,
 'has_precomputed_features': False,
 'loaded': True,
 'next_step': "Call featurize_dataset(dataset_id='chembl_activity_data_O00329_P48736', method='ECFP', radius=2, n_bits=2048) to compute fingerprints server-side, then split_prepared_dataset().",
 'smiles_sample': ['CC(NC(=O)c1c(N)nn2cccnc12)c1cc2cccc(Cl)c2c(=O)n1-c1ccc(O)cc1',
  'Cc1cccc(NS(=O)(=O)c2ccc(C)c(-c3cnc(N)c(-c4cnn(C)c4)c3)c2)n1',
  'Cc1ccc(S(=O)(=O)NCC(C)(C)O)cc1-c1cnc(N)c(-c2ncco2)c1']}

In [8]:
dataset_status(dataset_info["dataset_id"])

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'loaded': True,
 'raw_data': {'n_samples': 1277,
  'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
  'label_col': 'class_label',
  'smiles_col': 'smiles',
  'id_col': None},
 'prepared': False}

## 5. Featurize (server-side — no large arrays transferred)

In [9]:
featurized = compute_features(
    dataset_info["dataset_id"],
    method="ECFP",
    radius=2,
    n_bits=2048,
)
featurized

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'method': 'ECFP',
 'n_samples': 1277,
 'n_features': 2048,
 'prepared': True,
 'bit_info_saved': True,
 'next_step': "Call split_dataset('chembl_activity_data_O00329_P48736', train_size=0.7, val_size=0.0, test_size=0.3, stratified=True) to create splits."}

## 6. Split

In [10]:
data_splits = split_dataset(
    dataset_info["dataset_id"],
    train_size=0.6,
    val_size=0.0,
    test_size=0.4,
    stratified=True,
    split_type="random",
    save_path=str(_save_dir / "data_splits.csv")
)
data_splits

{'train': {'n_samples': 766,
  'indices': [699,
   1087,
   939,
   987,
   331,
   314,
   66,
   105,
   401,
   352,
   213,
   159,
   387,
   96,
   316,
   240,
   983,
   457,
   334,
   1177,
   206,
   1179,
   426,
   465,
   200,
   693,
   1257,
   1133,
   588,
   374,
   1175,
   202,
   230,
   1115,
   207,
   435,
   794,
   417,
   131,
   87,
   244,
   152,
   900,
   1230,
   390,
   1171,
   567,
   1240,
   565,
   226,
   869,
   1016,
   81,
   890,
   889,
   255,
   781,
   1094,
   582,
   347,
   99,
   15,
   419,
   755,
   471,
   11,
   573,
   450,
   430,
   497,
   908,
   1037,
   991,
   653,
   1106,
   409,
   50,
   367,
   702,
   220,
   1169,
   1131,
   504,
   775,
   1190,
   18,
   1117,
   392,
   632,
   1158,
   1275,
   1202,
   1039,
   1045,
   227,
   406,
   411,
   641,
   93,
   828,
   74,
   345,
   436,
   327,
   1046,
   640,
   1201,
   349,
   1211,
   929,
   1208,
   985,
   1261,
   866,
   773,
   286,
   518,
   712,

## 6b. Dataset & split visualisation

In [11]:
# MCP equivalent (one call):
#   plot_dataset_info(dataset_info["dataset_id"], split_file_path=data_splits["saved_to"])

_split = joblib.load(data_splits["saved_to"])
_all_labels = np.concatenate([_split["train_labels"], _split["test_labels"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_class_distribution(
    _all_labels,
    title="Class Distribution (full dataset)",
    ax=axes[0],
)
plot_split_statistics(
    data_splits["statistics"],
    title="Train / Test Split",
    ax=axes[1],
)
fig.tight_layout()
plt.show()

[03/27/26 15:12:41] INFO     Using categorical units to plot a list of strings that are all         category.py:224
                             parsable as floats or dates. If these strings should be plotted as                    
                             numbers, cast to the appropriate data type before plotting.                           

                    INFO     Using categorical units to plot a list of strings that are all         category.py:224
                             parsable as floats or dates. If these strings should be plotted as                    
                             numbers, cast to the appropriate data type before plotting.                           

C:\Users\janela\AppData\Local\Temp\ipykernel_26084\4195542558.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Train model (tune → train → evaluate)

**MCP pattern** (non-blocking, used by the LLM agent):
```python
job = train_model(split_file_path=..., algorithm="RFC", task="classification")
result = check_training(job["job_id"])   # poll until status != "running"
```

**Notebook cell below** uses the internal helper `build_model_from_split_file` (blocking, convenient for direct exploration).

In [12]:
# Non-blocking MCP approach (mirrors what the LLM agent does)
job = train_model(
    split_file_path=data_splits["saved_to"],
    algorithm="DNN",
    task="classification",
    opt_metric="balanced_accuracy",
    model_save_path=str(_save_dir / "trained_model.pkl")
)
print("Job submitted:", job["job_id"])

while True:
    status = check_training(job["job_id"])
    if status["status"] != "running":
        break
    print(f"Training... {status['elapsed_seconds']:.1f}s elapsed")
    time.sleep(5)

model_result = status["result"]
print("Status:", status["status"])
model_result

Job submitted: 439531e7-ebba-419c-8779-01a97d4ded4b
Training... 0.0s elapsed
Status: completed


{'algorithm': 'DNN',
 'task': 'classification',
 'cv_fold': 5,
 'opt_metric': 'balanced_accuracy',
 'best_params': {},
 'cv_best_score': None,
 'model_path': 'c:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\notebooks\\debug_outputs\\trained_model.pkl',
 'hyperparameters_searched': {},
 'train_evaluation': {'target': 'train',
  'algorithm': 'DNN',
  'overall_metrics': {'MCC': 1.0,
   'Accuracy': 1.0,
   'BA': 1.0,
   'F1 macro': 1.0,
   'F1 weighted': 1.0},
  'per_class_metrics': {'Class_0': {'Precision': 1.0,
    'Recall': 1.0,
    'F1': 1.0,
    'Support': 579},
   'Class_1': {'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0, 'Support': 158},
   'Class_2': {'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0, 'Support': 29}},
  'confusion_matrix': [[579, 0, 0], [0, 158, 0], [0, 0, 29]],
  'class_labels': [0, 1, 2]},
 'test_evaluation': {'target': 'test',
  'algorithm': 'DNN',
  'overall_metrics': {'MCC': 0.8625180911814769,
   'Accura

## 8. Inspect results

In [13]:
print("Best params  :", model_result["best_params"])
print("CV best score:", model_result["cv_best_score"])
print("Model saved  :", model_result["model_path"])

Best params  : {}
CV best score: None
Model saved  : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\notebooks\debug_outputs\trained_model.pkl


In [14]:
model_result["train_evaluation"]

{'target': 'train',
 'algorithm': 'DNN',
 'overall_metrics': {'MCC': 1.0,
  'Accuracy': 1.0,
  'BA': 1.0,
  'F1 macro': 1.0,
  'F1 weighted': 1.0},
 'per_class_metrics': {'Class_0': {'Precision': 1.0,
   'Recall': 1.0,
   'F1': 1.0,
   'Support': 579},
  'Class_1': {'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0, 'Support': 158},
  'Class_2': {'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0, 'Support': 29}},
 'confusion_matrix': [[579, 0, 0], [0, 158, 0], [0, 0, 29]],
 'class_labels': [0, 1, 2]}

In [15]:
model_result["test_evaluation"]

{'target': 'test',
 'algorithm': 'DNN',
 'overall_metrics': {'MCC': 0.8625180911814769,
  'Accuracy': 0.9471624266144814,
  'BA': 0.8727525628030385,
  'F1 macro': 0.8689431310388619,
  'F1 weighted': 0.947323482561949},
 'per_class_metrics': {'Class_0': {'Precision': 0.9638242894056848,
   'Recall': 0.966321243523316,
   'F1': 0.9650711513583441,
   'Support': 386},
  'Class_1': {'Precision': 0.9326923076923077,
   'Recall': 0.9150943396226415,
   'F1': 0.9238095238095239,
   'Support': 106},
  'Class_2': {'Precision': 0.7,
   'Recall': 0.7368421052631579,
   'F1': 0.717948717948718,
   'Support': 19}},
 'confusion_matrix': [[373, 7, 6], [9, 97, 0], [5, 0, 14]],
 'class_labels': [0, 1, 2]}

## 9. Standalone evaluation (optional)

Uses internal helpers `evaluate_classification` / `evaluate_regression` (not MCP tools — available for direct notebook use).

In [18]:
split    = joblib.load(data_splits["saved_to"])
model    = joblib.load(model_result["model_path"])
X_test   = np.array(split["test_features"])
y_test   = split["test_labels"].tolist()

y_pred   = model.predict(X_test).tolist()
y_proba  = model.predict_proba(X_test).tolist()
y_proba

[[1.0, 2.5623787247653177e-17, 8.401075264682104e-17],
 [4.09701124226558e-06, 0.9999959468841553, 2.4077750904183404e-09],
 [1.0, 1.8077192520671304e-11, 1.1918296211055335e-11],
 [4.573775640892563e-06, 0.9999954700469971, 1.1190056659060588e-09],
 [1.0, 3.5277166604563703e-11, 3.2166942720568414e-11],
 [1.0, 2.5133095257756644e-11, 2.8684861560698827e-13],
 [1.0, 4.89133789027818e-10, 1.3729765935277527e-10],
 [1.0, 2.1518203885406706e-10, 1.1075803607596413e-09],
 [3.017665869790853e-13, 1.0, 3.9134129185098513e-16],
 [0.024707769975066185, 0.975236177444458, 5.601003795163706e-05],
 [5.850035456056446e-10, 1.0, 1.1648466479580177e-12],
 [1.0, 3.4237999229096063e-15, 1.018642457642732e-15],
 [1.0, 1.5746545178885185e-09, 7.301249049795899e-10],
 [0.9999769926071167, 3.020110739271331e-07, 2.26199917960912e-05],
 [3.542944853052177e-07, 0.9999996423721313, 4.437922945799144e-11],
 [0.9999992847442627, 5.884805318601138e-07, 1.1109087694194386e-07],
 [1.0, 2.8925394523170167e-13, 1.1

## 9b. Classification plots

**MCP equivalent** (loads model + split from disk, generates all figures in one call):
```python
plot_classification_results(model_result["model_path"], data_splits["saved_to"])
```

Cells below call the underlying plot functions directly for finer notebook control.

In [16]:
_n_classes  = len(set(y_test))
_is_binary  = _n_classes == 2

# binary eval → flat dict; multiclass eval → nested, scalars in "overall_metrics"
_test_metrics = model_result["test_evaluation"]
_scalar_metrics = _test_metrics.get("overall_metrics", _test_metrics)

if _is_binary:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    plot_confusion_matrix(
        y_test, y_pred,
        title="Confusion Matrix (test)", ax=axes[0],
    )
    plot_roc_curve(
        y_test, [p[1] for p in y_proba],
        title="ROC Curve (test)", ax=axes[1],
    )
    plot_pr_curve(
        y_test, [p[1] for p in y_proba],
        title="Precision-Recall Curve (test)", ax=axes[2],
    )
else:
    # Multiclass: confusion matrix + overall metric bar
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_confusion_matrix(
        y_test, y_pred,
        title=f"Confusion Matrix — {_n_classes} classes (test)", ax=axes[0],
    )
    plot_metric_bar(_scalar_metrics, title="Test Set Metrics (overall)", ax=axes[1])

fig.tight_layout()
plt.show()

C:\Users\janela\AppData\Local\Temp\ipykernel_57296\2585638552.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Evaluation metrics bar chart
# binary eval → flat dict; multiclass eval → nested, scalars in "overall_metrics"
_eval = model_result["test_evaluation"]
_scalars = _eval.get("overall_metrics", _eval)
plot_metric_bar(_scalars, title="Test Set Metrics")
#plt.show()

## 9c. Export predictions to CSV (MCP tool)

`export_predictions` writes a per-sample CSV with (where available):

| Column | Description |
|---|---|
| `cid` | Compound ID (from the original dataset, if present) |
| `smiles` | SMILES string (if present) |
| `true_label` | Ground-truth label |
| `predicted_label` | Model prediction |
| `prob_class_0`, `prob_class_1` | Class probabilities (classification only) |

For regression tasks `predicted_label` is replaced by `predicted_value`. A `.pkl` metrics file is saved alongside the CSV. Both land in `<session>/results/`.

In [ ]:
export_result = export_predictions(
    model_path=model_result["model_path"],
    split_file_path=data_splits["saved_to"],
    task="classification",
    split="test",          # default
)
export_result

In [ ]:
import pandas as pd
pd.read_csv(export_result["csv_path"]).head(10)


## 10. Scratch / exploratory

In [13]:
from src.chemagent.servers.session_tools import _parse_chat_events

In [30]:
chat_events = _parse_chat_events(log_file=Path("./../data/logs/session_tiago_20260312_104021_8f6812/session_tiago_20260312_104021_8f6812.txt"))
chat_events

{'type': 'tool_call',
 'timestamp': '2026-03-12T09:43:08.112+00:00',
 'tool': 'split_dataset',
 'args': {'dataset_id': 'chembl_activity_data_O00329_P48736',
  'split_type': 'scaffold',
  'train_size': 0.8,
  'val_size': 0.0,
  'test_size': 0.2,
  'seed': 42,
  'stratified': False,
  'save_path': None},
 'status': 'success',
 'duration_ms': 545.81,
 'result': {'train': {'n_samples': 1021,
   'indices': {'__list_len__': 1021, '__sample__': [457, 380, 1182]},
   'smiles_sample': ['O=c1cc(N2CCOCC2)oc2c(-c3ccc4c(c3)OCO4)csc12',
    'CC(NC(=O)c1c(N)nn2cccnc12)c1cc2cccc(C#CC3CCCCC3)c2c(=O)n1-c1ccccc1',
    'O=C(c1ccccc1)c1ccc(N2CCOCC2)cc1O']},
  'val': {'n_samples': 0, 'indices': [], 'smiles_sample': []},
  'test': {'n_samples': 256,
   'indices': {'__list_len__': 256, '__sample__': [237, 365, 387]},
   'smiles_sample': ['Cc1nc(N)nc(N2CC(O)CC2c2nc3ccc(F)c(-c4cnn(C)c4)c3c(=O)n2-c2ccccc2)c1C#N',
    'Cc1ccccc1-n1c(C2CCCN2c2ncnc(N)c2C#N)nc2cccc(-c3cnn(C)c3)c2c1=O',
    'COC1CC(c2nc3cccc(-c4cnn(C

In [56]:
chat_events[4]

{'type': 'tool_call',
 'timestamp': '2026-03-12T09:45:00.938+00:00',
 'tool': 'get_ml_info',
 'args': {},
 'status': 'success',
 'duration_ms': 0.04,
 'result': {'algorithms': {'RFR': {'name': 'Random Forest Regressor',
    'task_type': 'regression',
    'hyperparameters': {'n_estimators': [50, 100, 200],
     'max_features': ['sqrt', 'log2'],
     'min_samples_split': [2, 5, 10],
     'min_samples_leaf': [1, 2, 4]},
    'supports_multiclass': False,
    'description': 'Ensemble of decision trees for regression tasks'},
   'RFC': {'name': 'Random Forest Classifier',
    'task_type': 'classification',
    'hyperparameters': {'n_estimators': [50, 100, 200],
     'max_features': ['sqrt', 'log2'],
     'min_samples_split': [2, 5, 10],
     'min_samples_leaf': [1, 2, 4]},
    'supports_multiclass': True,
    'description': 'Ensemble of decision trees for classification, handles multi-class'},
   'SVC': {'name': 'Support Vector Classifier',
    'task_type': 'classification',
    'hyperparame

In [26]:
thoughts = [ev for ev in chat_events if ev["type"] == "thought"]
thoughts

[{'type': 'thought',
  'timestamp': '2026-03-12T10:28:10.174+00:00',
  'step': 'summary',
  'thought': "Project Workflow Summary: This ML project successfully completed a full pipeline from chemical data to interpretable predictions. We imported the chembl_activity_data_O00329_P48736 dataset containing 1,277 compounds with SMILES structures and class labels (0, 1, 2). After computing 2,048-bit ECFP molecular fingerprints, we performed scaffold-based splitting (80% train, 20% test) to ensure chemical diversity between sets. A Random Forest Classifier was trained with 5-fold cross-validation and hyperparameter tuning, identifying optimal parameters: n_estimators=100, max_features='sqrt', min_samples_split=2, min_samples_leaf=1, achieving 83.6% CV balanced accuracy. The model achieved 68.1% balanced accuracy on the test set with strong performance on the majority class (Class 0: 93.0% F1). We generated SHAP explanations for a specific compound, revealing how its heterocyclic structure (py

In [3]:
from src.chemagent.servers.session_tools import log_answer, _parse_chat_events
from pathlib import Path

# 1) Record an assistant/LLM answer into the current session log
# (this writes to the active session file managed by SessionLogger)
log_answer("Test assistant reply: model suggests RFC.", role="assistant")

# 2) Read answers from an existing session log file (example path)
chat_events = _parse_chat_events(log_file=Path("./../data/logs/session_tiago_20260317_120356_8c47f0/session_tiago_20260317_120356_8c47f0.txt"))
answers = [ev for ev in chat_events if ev["type"] == "answer"]
answers

[{'type': 'answer',
  'timestamp': '2026-03-17T11:09:07.577+00:00',
  'role': 'assistant',
  'answer': 'Test assistant reply: model suggests RFC.'}]